# Unified Adversarial Patch Evaluation — All YOLO Models

Evaluates **YOLOv8n, YOLOv8s, YOLOv5n, RT-DETR** with metrics identical to `yolov8s_patch.ipynb`:
- **mAP50, mAP50-95, Precision, Recall** via `model.val()` (same conf=0.001, iou=0.6, imgsz=640)
- **Det/img and FP/img** at conf ≥ {0.001, 0.3, 0.5, 0.7} via `model.predict()` per image

**Two experiments per model:**
1. **Own-patch** — each billboard evaluated with its own optimised adversarial patch
2. **Patch1 transfer** — the billboard01 patch transferred to billboard02–09

---
### Key design decisions
| Decision | Rationale |
|---|---|
| `Clean` folder (original res) for images | YOLO applies letterbox internally — identical to `evaluation_ab.py`. Using pre-stretched 640×640 (`Clean_640`) caused 30–40% mAP collapse. |
| `model.val()` for mAP | Single-image `predict()` gives different mAP than the full dataset validation pipeline. |
| Class names from model | Avoids hardcoding errors; `evaluation_ab.py` does the same via `model.names`. |
| Same YAML for clean in own-patch and transfer | Guarantees clean mAP50 is bit-for-bit identical between the two experiments. |

In [ ]:
# ━━ CELL 1: Mount Drive & install ━━━━━━━━━━━━━━━━━━━━━━━━━━━
from google.colab import drive
import torch
drive.mount('/content/drive')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

!pip install ultralytics --quiet
import ultralytics
print('Ultralytics:', ultralytics.__version__)

In [ ]:
# ━━ CELL 2: Paths — edit these if your Drive layout differs ━━
import os
import numpy as np
import json
import csv
from pathlib import Path
from functools import partial

# ── Dataset roots ────────────────────────────────────────────
# CLEAN_ROOT: original-resolution images. YOLO letterboxes internally → correct mAP.
# DO NOT use Clean_640 here — that folder was created for NanoDet and uses a plain
# cv2.resize stretch that distorts objects and drops YOLO mAP by 30-40%.
CLEAN_ROOT     = '/content/drive/MyDrive/Patched Datasets/Clean'
PATCHED_PARENT = '/content/drive/MyDrive/Patched Datasets'
PATCH1_ROOT    = '/content/drive/MyDrive/Patch1BillX'   # Patch1Bill2 … Patch1Bill9

# ── Model checkpoints ────────────────────────────────────────
MODEL_PATHS = {
    'yolov8n': '/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt',
    'yolov8s': '/content/drive/MyDrive/yolov8s_carlagear/train2/weights/best.pt',
    'yolov5n': '/content/drive/MyDrive/yolov5n_carlagear/train/weights/best.pt',
    'rtdetr':  '/content/drive/MyDrive/runs/rtdetr_carlagear2/weights/best.pt',
}

# ── Output ───────────────────────────────────────────────────
OUTPUT_ROOT = '/content/drive/MyDrive/unified_eval'
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# ── Eval config — MUST match evaluation_ab.py in yolov8s_patch.ipynb ─
BILLBOARDS   = [f'billboard0{i}' for i in range(1, 10)]
THRESHOLDS   = [0.001, 0.3, 0.5, 0.7]
GT_IOU_MATCH = 0.5
CONF_MIN     = 0.001
IOU_NMS      = 0.6
IMGSZ        = 640
IMG_EXTS     = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
device       = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Verify ───────────────────────────────────────────────────
print('Model checkpoints:')
for name, path in MODEL_PATHS.items():
    ok   = Path(path).exists()
    size = f'{Path(path).stat().st_size/1e6:.1f} MB' if ok else 'MISSING'
    print(f'  {name:<12}  {"✅" if ok else "❌"}  {size}  {path}')

print('\nDataset roots:')
for label, path in [('Clean (YOLO)', CLEAN_ROOT), ('Patched', PATCHED_PARENT), ('Patch1', PATCH1_ROOT)]:
    ok = Path(path).exists()
    print(f'  {label:<14}  {"✅" if ok else "❌ MISSING"}  {path}')

In [ ]:
# ━━ CELL 3: Load models ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from ultralytics import YOLO, RTDETR

models = {}
for name in ['yolov8n', 'yolov8s', 'yolov5n']:
    models[name] = YOLO(MODEL_PATHS[name])
    print(f'{name} loaded  classes={list(models[name].names.values())[:4]}...')

models['rtdetr'] = RTDETR(MODEL_PATHS['rtdetr'])
print(f'rtdetr loaded  classes={list(models["rtdetr"].names.values())[:4]}...')
print('\nAll models loaded ✅')

In [ ]:
# ━━ CELL 4: Write dataset YAMLs with correct label pairing ━━
# KEY FIX: model.val() finds labels by replacing 'images' with 'labels' in the path.
# The patched_dataset{i}/labels/test/ files can differ from clean labels (different
# background counts, different instance counts) → wrong GT → wrong mAP.
#
# Solution: for every patched/transfer condition, create a temp directory that
# symlinks PATCHED IMAGES + CLEAN LABELS together, so model.val() always evaluates
# against the correct ground truth regardless of what's in patched_dataset labels.
#
# Directory structure created:
#   /tmp/eval_bb01_patched/
#       images/test/ → symlinks to patched_dataset1/images/test/*.png
#       labels/test/ → symlinks to Clean/labels/billboard01/*.txt
import yaml, os, shutil
from pathlib import Path

YAML_DIR = Path('/content/scenario_yamls')
TMP_ROOT  = Path('/tmp/eval_dirs')
YAML_DIR.mkdir(exist_ok=True)
TMP_ROOT.mkdir(exist_ok=True)

def get_class_names(model):
    names_dict = model.names
    return [names_dict[k] for k in sorted(names_dict.keys())]

CLASS_NAMES = get_class_names(models['yolov8n'])
NC          = len(CLASS_NAMES)
print(f'Classes ({NC}): {CLASS_NAMES}')

def write_yaml(root_dir: str, rel_img: str, out_path):
    """Write YAML where labels are found at root_dir/labels/... (replacing images → labels)."""
    data = {
        'path':  str(root_dir),
        'train': rel_img,
        'val':   rel_img,
        'test':  rel_img,
        'nc':    NC,
        'names': CLASS_NAMES,
    }
    with open(out_path, 'w') as f:
        yaml.safe_dump(data, f)

def make_eval_dir(img_src_dir: str, lab_src_dir: str, tag: str):
    """
    Create /tmp/eval_dirs/{tag}/
        images/test/  ← symlinks to img_src_dir images
        labels/test/  ← symlinks to lab_src_dir labels (clean GT)
    Returns the root path and the relative val path.
    """
    root    = TMP_ROOT / tag
    img_dst = root / 'images' / 'test'
    lab_dst = root / 'labels' / 'test'
    shutil.rmtree(root, ignore_errors=True)
    img_dst.mkdir(parents=True)
    lab_dst.mkdir(parents=True)

    # Symlink images from patched/transfer source
    for p in sorted(Path(img_src_dir).iterdir()):
        if p.suffix.lower() in IMG_EXTS:
            os.symlink(p.resolve(), img_dst / p.name)

    # Symlink labels from CLEAN source (correct GT)
    for p in sorted(Path(lab_src_dir).iterdir()):
        if p.suffix == '.txt':
            os.symlink(p.resolve(), lab_dst / p.name)

    return str(root), 'images/test'

# Write clean YAMLs (no symlink needed — clean images already have correct labels)
print('Writing YAMLs...')
n = 0
for i, bb in enumerate(BILLBOARDS, 1):
    clean_img = f'{CLEAN_ROOT}/images/billboard0{i}'
    clean_lab = f'{CLEAN_ROOT}/labels/billboard0{i}'

    # Clean: images and labels in same tree → direct YAML
    write_yaml(f'{CLEAN_ROOT}', f'images/billboard0{i}',
               YAML_DIR / f'{bb}_clean.yaml')
    n += 1

    # Patched (own patch): patched images + CLEAN labels via symlink dir
    patch_img = f'{PATCHED_PARENT}/patched_dataset{i}/images/test'
    root, rel = make_eval_dir(patch_img, clean_lab, f'bb{i:02d}_patched')
    write_yaml(root, rel, YAML_DIR / f'{bb}_patched.yaml')
    n += 1

    # Patch1 transfer: Patch1 images + CLEAN labels via symlink dir
    if i > 1:
        p1_img = Path(f'{PATCH1_ROOT}/Patch1Bill{i}/images/test')
        if p1_img.exists():
            root, rel = make_eval_dir(str(p1_img), clean_lab, f'bb{i:02d}_patch1')
            write_yaml(root, rel, YAML_DIR / f'{bb}_patch1.yaml')
            n += 1

print(f'Written {n} YAMLs, temp dirs in {TMP_ROOT}')

# Spot-check: verify bb01 patched dir has right images + labels
bb1_patch_img = list((TMP_ROOT / 'bb01_patched' / 'images' / 'test').iterdir())[:3]
bb1_patch_lab = list((TMP_ROOT / 'bb01_patched' / 'labels' / 'test').iterdir())[:3]
print(f'\nbb01_patched: {len(list((TMP_ROOT / "bb01_patched" / "images" / "test").iterdir()))} images, ')
print(f'              {len(list((TMP_ROOT / "bb01_patched" / "labels" / "test").iterdir()))} label files')
print(f'Sample image symlink: {bb1_patch_img[0]} → {os.readlink(bb1_patch_img[0])}')
print(f'Sample label symlink: {bb1_patch_lab[0]} → {os.readlink(bb1_patch_lab[0])}')

with open(YAML_DIR / 'billboard01_clean.yaml') as f:
    print(f'\nSample clean YAML:\n{f.read()}')
with open(YAML_DIR / 'billboard01_patched.yaml') as f:
    print(f'Sample patched YAML (uses symlink dir):\n{f.read()}')

In [ ]:
# ━━ CELL 5: Shared evaluation helpers ━━━━━━━━━━━━━━━━━━━━━━━
IMG_SIZE = 640   # GT coordinate space (matches model.val imgsz)

# ── GT labels ────────────────────────────────────────────────
def read_gt(txt_path, img_size=IMG_SIZE):
    out, p = [], Path(txt_path)
    if not p.exists(): return out
    for ln in p.read_text(errors='ignore').strip().splitlines():
        parts = ln.strip().split()
        if len(parts) < 5: continue
        c = int(float(parts[0]))
        cx, cy, bw, bh = map(float, parts[1:5])
        out.append((c, (cx-bw/2)*img_size, (cy-bh/2)*img_size,
                       (cx+bw/2)*img_size, (cy+bh/2)*img_size))
    return out

# ── IoU ──────────────────────────────────────────────────────
def iou_xyxy(a, b):
    ix1,iy1 = max(a[0],b[0]), max(a[1],b[1])
    ix2,iy2 = min(a[2],b[2]), min(a[3],b[3])
    inter   = max(0,ix2-ix1)*max(0,iy2-iy1)
    union   = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/union if union > 0 else 0.0

# ── predict wrapper (single image, same settings as model.val) ─
def predict_fn(model, img_path):
    """Returns (boxes[N,4] xyxy in [0,640], labels[N] 0-indexed, scores[N])."""
    r = model.predict(str(img_path), imgsz=IMGSZ, conf=CONF_MIN,
                      iou=IOU_NMS, device=device, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        return (np.zeros((0,4),np.float32), np.zeros((0,),np.int32), np.zeros((0,),np.float32))
    return (r.boxes.xyxy.cpu().numpy().astype(np.float32),
            r.boxes.cls.cpu().numpy().astype(np.int32),
            r.boxes.conf.cpu().numpy().astype(np.float32))

# ── mAP50 via model.val() ─────────────────────────────────────
def compute_map(model, yaml_path):
    r = model.val(data=str(yaml_path), imgsz=IMGSZ, conf=CONF_MIN,
                  iou=IOU_NMS, device=device, verbose=False, plots=False, save=False)
    return {'mAP50':     round(float(r.box.map50), 4),
            'mAP50_95':  round(float(r.box.map),   4),
            'precision': round(float(r.box.mp),    4),
            'recall':    round(float(r.box.mr),    4)}

# ── Det/img + FP/img at all thresholds ────────────────────────
def compute_density(model, img_dir, lab_dir):
    imgs = sorted([f for f in Path(img_dir).iterdir() if f.suffix.lower() in IMG_EXTS])
    if not imgs: return None
    det_t = {t: [] for t in THRESHOLDS}
    fp_t  = {t: [] for t in THRESHOLDS}
    for img_path in imgs:
        boxes, labels, scores = predict_fn(model, img_path)
        gt      = read_gt(Path(lab_dir) / f'{img_path.stem}.txt')
        gt_cls  = {}
        for (c,x1,y1,x2,y2) in gt: gt_cls.setdefault(c,[]).append((x1,y1,x2,y2))
        for t in THRESHOLDS:
            mask = scores >= t
            pf   = sorted(zip(labels[mask], boxes[mask], scores[mask]), key=lambda z:-z[2])
            det_t[t].append(len(pf))
            rem, n_fp = {c: list(v) for c,v in gt_cls.items()}, 0
            for c, box, _ in pf:
                best_iou, best_j = 0.0, None
                for j, gb in enumerate(rem.get(int(c), [])):
                    v = iou_xyxy(box, gb)
                    if v > best_iou: best_iou, best_j = v, j
                if best_j is not None and best_iou >= GT_IOU_MATCH:
                    rem[int(c)].pop(best_j)
                else:
                    n_fp += 1
            fp_t[t].append(n_fp)
    return {'det_per_img': {str(t): round(float(np.mean(det_t[t])),4) for t in THRESHOLDS},
             'fp_per_img': {str(t): round(float(np.mean(fp_t[t])), 4) for t in THRESHOLDS}}

# ── Full scenario eval ─────────────────────────────────────────
def eval_scenario(model, clean_yaml, patch_yaml, clean_img, clean_lab, patch_img, patch_lab):
    print(f'    val clean   ...', end=' ', flush=True)
    c_map = compute_map(model, clean_yaml);  print(f'mAP50={c_map["mAP50"]:.4f}')
    print(f'    val patched ...', end=' ', flush=True)
    p_map = compute_map(model, patch_yaml);  print(f'mAP50={p_map["mAP50"]:.4f}')
    print(f'    density clean  ...', flush=True)
    c_den = compute_density(model, clean_img, clean_lab)
    print(f'    density patched...', flush=True)
    p_den = compute_density(model, patch_img, patch_lab)
    c = {**c_map, **(c_den or {})}
    p = {**p_map, **(p_den or {})}
    d = {
        'mAP50':     round(p_map['mAP50']     - c_map['mAP50'],    4),
        'mAP50_95':  round(p_map['mAP50_95']  - c_map['mAP50_95'], 4),
        'precision': round(p_map['precision'] - c_map['precision'], 4),
        'recall':    round(p_map['recall']    - c_map['recall'],    4),
        'det_per_img': {str(t): round(
            (p_den or {}).get('det_per_img',{}).get(str(t),0) -
            (c_den or {}).get('det_per_img',{}).get(str(t),0), 4) for t in THRESHOLDS},
        'fp_per_img':  {str(t): round(
            (p_den or {}).get('fp_per_img',{}).get(str(t),0) -
            (c_den or {}).get('fp_per_img',{}).get(str(t),0), 4) for t in THRESHOLDS},
    }
    return {'clean': c, 'patched': p, 'delta': d}

# ── Summary print ──────────────────────────────────────────────
def print_summary(res, title):
    bbs  = [b for b in BILLBOARDS if b in res]
    c50  = [res[b]['clean']['mAP50']   for b in bbs]
    p50  = [res[b]['patched']['mAP50'] for b in bbs]
    d50  = [res[b]['delta']['mAP50']   for b in bbs]
    print(f'\n{"═"*80}')
    print(f'{title}')
    print(f'{"-"*80}')
    print(f'{"Scenario":<14} {"C-mAP":>7} {"P-mAP":>7} {"Δ-mAP":>8}  '
          f'{"DetC@.001":>10} {"DetP@.001":>10}  {"FPC@.5":>7} {"FPP@.5":>7}')
    print(f'{"-"*80}')
    for bb in bbs:
        r   = res[bb]
        cd1 = r['clean'].get('det_per_img',{}).get('0.001',0)
        pd1 = r['patched'].get('det_per_img',{}).get('0.001',0)
        cf5 = r['clean'].get('fp_per_img',{}).get('0.5',0)
        pf5 = r['patched'].get('fp_per_img',{}).get('0.5',0)
        print(f'{bb:<14} {r["clean"]["mAP50"]:>7.4f} {r["patched"]["mAP50"]:>7.4f} '
              f'{r["delta"]["mAP50"]:>+8.4f}  {cd1:>10.2f} {pd1:>10.2f}  {cf5:>7.2f} {pf5:>7.2f}')
    print(f'{"-"*80}')
    print(f'{"Mean":<14} {np.mean(c50):>7.4f} {np.mean(p50):>7.4f} '
          f'{np.mean(d50):>+8.4f} ±{np.std(d50):.4f}')

# ── Save JSON + CSV ────────────────────────────────────────────
def save_results(res, run_name):
    out = Path(OUTPUT_ROOT) / run_name
    out.mkdir(parents=True, exist_ok=True)
    with open(out / 'results.json', 'w') as f:
        json.dump(res, f, indent=2)
    rows = []
    for bb, r in res.items():
        row = {'Scenario': bb, 'Variant': run_name,
               'mAP50_C': r['clean']['mAP50'],       'mAP50_P': r['patched']['mAP50'],
               'mAP50_Δ': r['delta']['mAP50'],
               'mAP50_95_C': r['clean'].get('mAP50_95',''),
               'mAP50_95_P': r['patched'].get('mAP50_95',''),
               'mAP50_95_Δ': r['delta'].get('mAP50_95',''),
               'Precision_C': r['clean'].get('precision',''),
               'Precision_P': r['patched'].get('precision',''),
               'Precision_Δ': r['delta'].get('precision',''),
               'Recall_C': r['clean'].get('recall',''),
               'Recall_P': r['patched'].get('recall',''),
               'Recall_Δ': r['delta'].get('recall','')}
        for t in THRESHOLDS:
            row[f'Det/img@{t}_C'] = r['clean'].get('det_per_img',{}).get(str(t),'')
            row[f'Det/img@{t}_P'] = r['patched'].get('det_per_img',{}).get(str(t),'')
            row[f'Det/img@{t}_Δ'] = r['delta'].get('det_per_img',{}).get(str(t),'')
            row[f'FP/img@{t}_C']  = r['clean'].get('fp_per_img',{}).get(str(t),'')
            row[f'FP/img@{t}_P']  = r['patched'].get('fp_per_img',{}).get(str(t),'')
            row[f'FP/img@{t}_Δ']  = r['delta'].get('fp_per_img',{}).get(str(t),'')
        rows.append(row)
    with open(out / 'results.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)
    print(f'  Saved → {out}')

print('Helpers ready ✅')

## Experiment 1 — Own-patch evaluation
Each billboard scene is evaluated against its own adversarial patch (`patched_dataset{i}`).

In [ ]:
# ━━ CELL 6: Own-patch evaluation — all YOLO models ━━━━━━━━━━━
own_results = {}   # {model_name: {bb: {clean, patched, delta}}}

for model_name, model in models.items():
    print(f'\n{"═"*60}\nModel: {model_name}\n{"═"*60}')
    own_results[model_name] = {}

    for i, bb in enumerate(BILLBOARDS, 1):
        print(f'\n  ── {bb} ──')
        clean_yaml   = YAML_DIR / f'{bb}_clean.yaml'
        patched_yaml = YAML_DIR / f'{bb}_patched.yaml'
        clean_img    = f'{CLEAN_ROOT}/images/billboard0{i}'
        clean_lab    = f'{CLEAN_ROOT}/labels/billboard0{i}'
        patch_img    = f'{PATCHED_PARENT}/patched_dataset{i}/images/test'

        r = eval_scenario(model, clean_yaml, patched_yaml,
                          clean_img, clean_lab, patch_img, clean_lab)
        own_results[model_name][bb] = r
        print(f'    → Δ mAP50={r["delta"]["mAP50"]:+.4f}  '
              f'DetC@001={r["clean"].get("det_per_img",{}).get("0.001",0):.1f}  '
              f'DetP@001={r["patched"].get("det_per_img",{}).get("0.001",0):.1f}  '
              f'FPC@5={r["clean"].get("fp_per_img",{}).get("0.5",0):.2f}  '
              f'FPP@5={r["patched"].get("fp_per_img",{}).get("0.5",0):.2f}')

    print_summary(own_results[model_name], f'{model_name} — Own-patch')
    save_results(own_results[model_name], f'{model_name}_own_patch')

print('\n✅ Own-patch evaluation complete for all models')

## Experiment 2 — Cross-scenario Patch1 transfer
The patch optimised on billboard01 is applied to billboard02–09.
The clean evaluation reuses the **same YAML files** as Experiment 1,
so clean mAP50 will be **bit-for-bit identical** — no dataset mismatch possible.

In [ ]:
# ━━ CELL 7: Patch1 transfer — all YOLO models ━━━━━━━━━━━━━━━━
transfer_results = {}   # {model_name: {bb: {clean, patched, delta}}}
TARGET_BBs = list(range(2, 10))

# Verify transfer images exist before we start
print('Transfer image availability:')
for i in TARGET_BBs:
    p = Path(f'{PATCH1_ROOT}/Patch1Bill{i}/images/test')
    n = len(list(p.glob('*'))) if p.exists() else 0
    print(f'  billboard0{i}: {"✅" if p.exists() else "❌ MISSING"}  ({n} images)  {p}')

print()
for model_name, model in models.items():
    print(f'\n{"═"*60}\nModel: {model_name}\n{"═"*60}')
    transfer_results[model_name] = {}

    for i in TARGET_BBs:
        bb           = f'billboard0{i}'
        patch1_yaml  = YAML_DIR / f'{bb}_patch1.yaml'
        clean_yaml   = YAML_DIR / f'{bb}_clean.yaml'   # ← SAME as own-patch clean
        clean_img    = f'{CLEAN_ROOT}/images/billboard0{i}'
        clean_lab    = f'{CLEAN_ROOT}/labels/billboard0{i}'
        patch_img    = f'{PATCH1_ROOT}/Patch1Bill{i}/images/test'

        if not Path(patch_img).exists():
            print(f'  ── {bb}: ⚠️  patch images missing, skipping')
            continue
        if not patch1_yaml.exists():
            print(f'  ── {bb}: ⚠️  YAML missing — re-run Cell 4')
            continue

        print(f'\n  ── {bb} (Patch1) ──')
        r = eval_scenario(model, clean_yaml, patch1_yaml,
                          clean_img, clean_lab, patch_img, clean_lab)
        transfer_results[model_name][bb] = r
        print(f'    → Clean={r["clean"]["mAP50"]:.4f}  '
              f'Patched={r["patched"]["mAP50"]:.4f}  '
              f'Δ={r["delta"]["mAP50"]:+.4f}')

    if transfer_results[model_name]:
        print_summary(transfer_results[model_name], f'{model_name} — Patch1 transfer')
        save_results(transfer_results[model_name], f'{model_name}_patch1_transfer')

print('\n✅ Patch1 transfer evaluation complete for all models')

In [ ]:
# ━━ CELL 8: Save detection images for Patch1 transfer ━━━━━━━━━━━━━━━━━━━━━
import cv2
from pathlib import Path

# Output folder
VIS_ROOT = Path(OUTPUT_ROOT) / "detection_images_patch1_transfer"
VIS_ROOT.mkdir(parents=True, exist_ok=True)

# Detection visualization settings
CONF_TO_DRAW = 0.3          # change to 0.001 if you want to see almost everything
MAX_IMAGES_PER_SCENARIO = None   # set to e.g. 20 for quick testing, or None for all images

def list_images(img_dir):
    img_dir = Path(img_dir)
    if not img_dir.exists():
        return []
    imgs = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS])
    if MAX_IMAGES_PER_SCENARIO is not None:
        imgs = imgs[:MAX_IMAGES_PER_SCENARIO]
    return imgs

def save_detection_images(model, model_name, bb, variant_name, img_dir):
    """
    Saves images with predicted detection boxes drawn on them.

    Output:
      OUTPUT_ROOT/detection_images_patch1_transfer/
          yolov8n/
              billboard02/
                  clean/
                  patch1/
    """
    img_paths = list_images(img_dir)

    if len(img_paths) == 0:
        print(f"⚠️ No images found: {img_dir}")
        return

    out_dir = VIS_ROOT / model_name / bb / variant_name
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"Saving {model_name} | {bb} | {variant_name} → {out_dir}")

    for img_path in img_paths:
        result = model.predict(
            str(img_path),
            imgsz=IMGSZ,
            conf=CONF_TO_DRAW,
            iou=IOU_NMS,
            device=device,
            verbose=False
        )[0]

        # Ultralytics draws boxes, labels, confidences
        annotated = result.plot()

        # Save as jpg
        save_path = out_dir / f"{img_path.stem}_det.jpg"
        cv2.imwrite(str(save_path), annotated)

    print(f"  ✅ Saved {len(img_paths)} images")


# ── Run for all models and Patch1 transfer scenarios ─────────────
TARGET_BBs = list(range(2, 10))

for model_name, model in models.items():
    print(f"\n{'═' * 70}")
    print(f"Saving detections for model: {model_name}")
    print(f"{'═' * 70}")

    for i in TARGET_BBs:
        bb = f"billboard0{i}"

        clean_img_dir = f"{CLEAN_ROOT}/images/billboard0{i}"
        patch1_img_dir = f"{PATCH1_ROOT}/Patch1Bill{i}/images/test"

        if not Path(patch1_img_dir).exists():
            print(f"⚠️ Skipping {bb}: Patch1 images missing")
            continue

        # Clean detections
        save_detection_images(
            model=model,
            model_name=model_name,
            bb=bb,
            variant_name="clean",
            img_dir=clean_img_dir
        )

        # Patch1 transfer detections
        save_detection_images(
            model=model,
            model_name=model_name,
            bb=bb,
            variant_name="patch1",
            img_dir=patch1_img_dir
        )

print(f"\n✅ All detection images saved to:\n{VIS_ROOT}")

In [ ]:
# ━━ CELL 8: Sanity check ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Clean mAP50 must be IDENTICAL between own-patch and transfer (same YAML, same images).
# Any non-zero difference indicates a path or YAML mismatch.
print(f'{"Model":<12} {"Scenario":<14} {"Own-patch clean":>16} {"Transfer clean":>15} {"Δ":>8} {"OK?":>5}')
print('-'*72)
all_ok = True
for mn in models:
    for i in TARGET_BBs:
        bb  = f'billboard0{i}'
        own = own_results.get(mn,{}).get(bb,{}).get('clean',{}).get('mAP50')
        tra = transfer_results.get(mn,{}).get(bb,{}).get('clean',{}).get('mAP50')
        if own is None or tra is None: continue
        diff = abs(own - tra)
        ok   = diff < 1e-4
        if not ok: all_ok = False
        print(f'{mn:<12} {bb:<14} {own:>16.4f} {tra:>15.4f} {tra-own:>+8.4f} {"✅" if ok else "❌ MISMATCH":>5}')
print()
print('✅ All clean mAP values match — no dataset mismatch.' if all_ok
      else '❌ Mismatch detected — check that clean_yaml paths are identical in Cells 6 & 7.')

In [ ]:
# ━━ CELL 9: Cross-model comparison ━━━━━━━━━━━━━━━━━━━━━━━━━━━
PARAMS = {'yolov8n': '3.2M', 'yolov8s': '11.2M', 'yolov5n': '1.9M', 'rtdetr': '32.8M'}

for exp_name, exp_res in [('Own-patch (BB01–09)', own_results),
                           ('Patch1 transfer (BB02–09)', transfer_results)]:
    print(f'\n{"═"*90}')
    print(f'{exp_name}')
    print(f'{"-"*90}')
    print(f'{"Model":<14} {"Params":>7}  {"Clean mAP":>10} {"Patch mAP":>10} '
          f'{"Δ mAP50":>9} {"±std":>6}  {"CleanFP@.5":>11} {"PatchFP@.5":>11}')
    print(f'{"-"*90}')
    rows = []
    for mn in models:
        res = exp_res.get(mn, {})
        if not res: continue
        bbs  = [b for b in BILLBOARDS if b in res]
        c50  = [res[b]['clean']['mAP50']                         for b in bbs]
        p50  = [res[b]['patched']['mAP50']                       for b in bbs]
        d50  = [res[b]['delta']['mAP50']                         for b in bbs]
        cf5  = [res[b]['clean'].get('fp_per_img',{}).get('0.5',0) for b in bbs]
        pf5  = [res[b]['patched'].get('fp_per_img',{}).get('0.5',0) for b in bbs]
        print(f'{mn:<14} {PARAMS.get(mn,"?"):>7}  '
              f'{np.mean(c50):>10.4f} {np.mean(p50):>10.4f} '
              f'{np.mean(d50):>+9.4f} {np.std(d50):>6.4f}  '
              f'{np.mean(cf5):>11.4f} {np.mean(pf5):>11.4f}')
        rows.append({'exp': exp_name, 'model': mn, 'params': PARAMS.get(mn,'?'),
                     'clean_mAP50': round(np.mean(c50),4), 'patch_mAP50': round(np.mean(p50),4),
                     'delta_mean':  round(np.mean(d50),4), 'delta_std':   round(np.std(d50),4),
                     'clean_fp05':  round(np.mean(cf5),4), 'patch_fp05':  round(np.mean(pf5),4)})
    print(f'{"-"*90}')
    if rows:
        with open(f'{OUTPUT_ROOT}/{exp_name.split("(")[0].strip().replace(" ","_")}_comparison.csv',
                  'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader(); w.writerows(rows)

print(f'\nAll CSVs saved to {OUTPUT_ROOT}')
print('✅ Evaluation complete')